In [ ]:
import os
from getpass import getpass

kaggle_token = getpass("Paste Kaggle API here")

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(kaggle_token.strip())
os.chmod('/root/.kaggle/access_token', 0o600)

print("Kaggle API Setup Done ")

## Step 2: Libraries Install & Import 

In [ ]:
!pip install kaggle -q

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import pickle

import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded !")

## Step 3: Dataset Download

Data **Fake and Real News Dataset**  `Fake.csv` and `True.csv`.

In [ ]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p /content/data --unzip
print("Dataset download ho gaya ✅")
!ls /content/data

## Step 4: Data Load and Label 

In [ ]:
fake_df = pd.read_csv('/content/data/Fake.csv')
real_df = pd.read_csv('/content/data/True.csv')

fake_df['label'] = 1   # 1 = FAKE
real_df['label'] = 0   # 0 = REAL

print("Fake news:", fake_df.shape)
print("Real news:", real_df.shape)

df = pd.concat([fake_df, real_df], axis=0).reset_index(drop=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)   # shuffle

df['content'] = df['title'].fillna('') + ' ' + df['text'].fillna('')

print("\nTotal rows:", df.shape)
df[['title', 'label']].head()

In [ ]:

print(df['label'].value_counts())
sns.countplot(x='label', data=df)
plt.xticks([0, 1], ['Real (0)', 'Fake (1)'])
plt.title('Class Distribution')
plt.show()

## Step 5: Text Cleaning / Preprocessing (NLP)



In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)          # URLs
    text = re.sub(r'<.*?>', ' ', text)                          # HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), ' ', text)  # punctuation
    text = re.sub(r'\d+', ' ', text)                            # numbers
    text = re.sub(r'\s+', ' ', text).strip()                    # extra spaces
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return ' '.join(words)

print("Text cleaning shuru... (thoda time lagega, dataset bada hai)")
df['clean_content'] = df['content'].apply(clean_text)
print("Cleaning complete")

df[['content', 'clean_content']].head(3)

## Step 6: Train/Test Split + TF-IDF Vectorization

In [ ]:
X = df['clean_content']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1, 2), min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

## Step 7: Multiple Models Train and Compare

use three model and among them will use more efficient model 
1. **Logistic Regression**
2. **Passive Aggressive Classifier**
3. **Multinomial Naive Bayes**


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=5),
    "Passive Aggressive Classifier": PassiveAggressiveClassifier(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB()
}

results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    trained_models[name] = model
    print(f"{name}: {acc*100:.2f}% accuracy")
    print(classification_report(y_test, preds, target_names=['Real', 'Fake']))
    print("-" * 60)

In [ ]:
best_model_name = max(results, key=results.get)
best_model = trained_models[best_model_name]

print(f" Best model: {best_model_name} ({results[best_model_name]*100:.2f}% accuracy)")

# Accuracy comparison chart
plt.figure(figsize=(7, 4))
sns.barplot(x=list(results.keys()), y=list(results.values()))
plt.ylabel('Accuracy')
plt.title('Model Comparison')
plt.xticks(rotation=15)
plt.ylim(0.8, 1.0)
plt.show()

In [ ]:
best_preds = best_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, best_preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model_name}')
plt.show()

## Step 8: Model and Vectorizer

In [ ]:
with open('/content/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('/content/vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('/content/model_info.pkl', 'wb') as f:
    pickle.dump({'name': best_model_name, 'accuracy': results[best_model_name]}, f)

print(f"Model saved: {best_model_name} ✅")
print("Files: model.pkl, vectorizer.pkl, model_info.pkl")

## Step 9: Quick Test

In [ ]:
def predict_news(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = best_model.predict(vec)[0]

    if hasattr(best_model, "predict_proba"):
        confidence = best_model.predict_proba(vec)[0][pred]
    elif hasattr(best_model, "decision_function"):
        score = best_model.decision_function(vec)[0]
        confidence = 1 / (1 + np.exp(-abs(score)))   # sigmoid approx
    else:
        confidence = 1.0

    label = "FAKE" if pred == 1 else "REAL"
    return label, round(float(confidence) * 100, 2)

sample_text = "Scientists confirm the earth is flat and NASA has been lying for decades, insider reveals shocking truth."
print(predict_news(sample_text))

## Step 10: HTML + CSS + JavaScript Interface


In [ ]:
import os
os.makedirs('/content/templates', exist_ok=True)
os.makedirs('/content/static', exist_ok=True)
print("Folders ready ✅")

### `templates/index.html`

In [ ]:
%%writefile /content/templates/index.html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <title>Fake News Detector</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}" />
</head>
<body>
  <header class="masthead">
    <p class="kicker">AI-Powered &middot; Machine Learning</p>
    <h1>THE FAKE NEWS DETECTOR</h1>
    <p class="tagline">Paste any news article below and let the model analyze its authenticity</p>
    <div class="rule"></div>
  </header>

  <main class="wrapper">
    <section class="input-panel">
      <label for="articleText">News Article / Headline</label>
      <textarea id="articleText" rows="10" placeholder="Paste the news headline or full article text here..."></textarea>
      <button id="analyzeBtn">Analyze Article</button>
      <p id="statusMsg"></p>
    </section>

    <section id="resultPanel" class="result-panel hidden">
      <div class="verdict-row">
        <div id="verdictBadge" class="verdict-badge">—</div>
        <div class="confidence-block">
          <span class="confidence-label">Model Confidence</span>
          <div class="confidence-bar">
            <div id="confidenceFill" class="confidence-fill"></div>
          </div>
          <span id="confidenceText" class="confidence-text">0%</span>
        </div>
      </div>

      <div class="keywords-block">
        <h3>Influential Words in This Prediction</h3>
        <div id="keywordChips" class="chips"></div>
      </div>

      <p class="disclaimer">
        ⚠️ This is a statistical language-pattern model trained on a specific news dataset — it is
        <strong>not a fact-checking service</strong>. Always verify important news through trusted, primary sources.
      </p>
    </section>
  </main>

  <footer class="footer">
    <p>Built with TF-IDF + Machine Learning &middot; For educational purposes only</p>
  </footer>

  <script src="{{ url_for('static', filename='script.js') }}"></script>
</body>
</html>

### `static/style.css` — newspaper/editorial theme

In [ ]:
%%writefile /content/static/style.css
* { box-sizing: border-box; }

body {
  margin: 0;
  font-family: Georgia, 'Times New Roman', serif;
  background: #f4f1ea;
  color: #1a1a1a;
  min-height: 100vh;
}

.masthead {
  text-align: center;
  padding: 40px 20px 20px;
  border-bottom: 4px double #1a1a1a;
  background: #fdfcf8;
}

.kicker {
  text-transform: uppercase;
  letter-spacing: 3px;
  font-size: 0.75rem;
  color: #8b0000;
  font-weight: bold;
  margin: 0 0 6px;
}

.masthead h1 {
  margin: 0;
  font-size: 2.6rem;
  letter-spacing: 2px;
  font-weight: 900;
}

.tagline {
  color: #555;
  font-style: italic;
  margin: 10px 0 0;
}

.rule {
  height: 2px;
  background: #1a1a1a;
  margin: 18px auto 0;
  max-width: 700px;
}

.wrapper {
  max-width: 800px;
  margin: 40px auto;
  padding: 0 20px;
}

.input-panel {
  background: #fdfcf8;
  border: 1px solid #d8d3c4;
  border-radius: 6px;
  padding: 26px;
  box-shadow: 0 4px 14px rgba(0,0,0,0.06);
}

.input-panel label {
  display: block;
  font-weight: bold;
  margin-bottom: 10px;
  text-transform: uppercase;
  font-size: 0.8rem;
  letter-spacing: 1px;
  color: #444;
}

textarea {
  width: 100%;
  padding: 14px;
  font-family: Georgia, serif;
  font-size: 1rem;
  border: 1.5px solid #cfc9b8;
  border-radius: 4px;
  background: #fffef9;
  color: #1a1a1a;
  resize: vertical;
  outline: none;
}

textarea:focus {
  border-color: #8b0000;
}

button {
  margin-top: 16px;
  width: 100%;
  padding: 15px;
  border: none;
  border-radius: 4px;
  background: #8b0000;
  color: #fdfcf8;
  font-family: Georgia, serif;
  font-weight: bold;
  font-size: 1rem;
  letter-spacing: 1px;
  text-transform: uppercase;
  cursor: pointer;
  transition: background 0.15s ease, transform 0.1s ease;
}

button:hover { background: #6d0000; transform: translateY(-1px); }
button:disabled { opacity: 0.6; cursor: not-allowed; }

#statusMsg {
  text-align: center;
  min-height: 20px;
  color: #8b0000;
  font-style: italic;
  margin: 12px 0 0;
}

.result-panel {
  margin-top: 30px;
  background: #fdfcf8;
  border: 1px solid #d8d3c4;
  border-radius: 6px;
  padding: 26px;
  box-shadow: 0 4px 14px rgba(0,0,0,0.06);
}

.hidden { display: none; }

.verdict-row {
  display: flex;
  align-items: center;
  gap: 24px;
  flex-wrap: wrap;
}

.verdict-badge {
  font-size: 1.4rem;
  font-weight: 900;
  letter-spacing: 2px;
  padding: 16px 26px;
  border-radius: 6px;
  text-transform: uppercase;
  color: #fdfcf8;
  background: #555;
  min-width: 130px;
  text-align: center;
}

.verdict-badge.real { background: #1e5e2b; }
.verdict-badge.fake { background: #8b0000; }

.confidence-block {
  flex: 1;
  min-width: 200px;
}

.confidence-label {
  display: block;
  font-size: 0.8rem;
  text-transform: uppercase;
  letter-spacing: 1px;
  color: #555;
  margin-bottom: 6px;
}

.confidence-bar {
  width: 100%;
  height: 14px;
  background: #e5e0d3;
  border-radius: 20px;
  overflow: hidden;
  border: 1px solid #d8d3c4;
}

.confidence-fill {
  height: 100%;
  width: 0%;
  background: linear-gradient(90deg, #8b0000, #c0392b);
  transition: width 0.5s ease;
}

.confidence-text {
  display: block;
  margin-top: 6px;
  font-weight: bold;
}

.keywords-block {
  margin-top: 28px;
  border-top: 1px dashed #cfc9b8;
  padding-top: 20px;
}

.keywords-block h3 {
  margin: 0 0 12px;
  font-size: 1rem;
  text-transform: uppercase;
  letter-spacing: 1px;
}

.chips {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
}

.chip {
  background: #efe9d8;
  border: 1px solid #d8d3c4;
  padding: 5px 12px;
  border-radius: 20px;
  font-size: 0.85rem;
}

.disclaimer {
  margin-top: 22px;
  font-size: 0.8rem;
  color: #666;
  font-style: italic;
  line-height: 1.5;
}

.footer {
  text-align: center;
  padding: 30px 20px 50px;
  color: #777;
  font-size: 0.8rem;
}

### `static/script.js`

In [ ]:
%%writefile /content/static/script.js
const articleText = document.getElementById('articleText');
const analyzeBtn = document.getElementById('analyzeBtn');
const statusMsg = document.getElementById('statusMsg');
const resultPanel = document.getElementById('resultPanel');
const verdictBadge = document.getElementById('verdictBadge');
const confidenceFill = document.getElementById('confidenceFill');
const confidenceText = document.getElementById('confidenceText');
const keywordChips = document.getElementById('keywordChips');

analyzeBtn.addEventListener('click', async () => {
  const text = articleText.value.trim();
  if (!text) {
    statusMsg.textContent = 'Pehle koi article ya headline paste karein.';
    return;
  }

  statusMsg.textContent = 'Analyzing...';
  analyzeBtn.disabled = true;
  resultPanel.classList.add('hidden');

  try {
    const res = await fetch('/api/predict', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ text })
    });
    const data = await res.json();

    if (data.error) {
      statusMsg.textContent = data.error;
      return;
    }

    statusMsg.textContent = '';
    verdictBadge.textContent = data.label;
    verdictBadge.className = 'verdict-badge ' + (data.label === 'REAL' ? 'real' : 'fake');

    confidenceFill.style.width = data.confidence + '%';
    confidenceText.textContent = data.confidence + '%';

    keywordChips.innerHTML = '';
    (data.keywords || []).forEach(word => {
      const chip = document.createElement('span');
      chip.className = 'chip';
      chip.textContent = word;
      keywordChips.appendChild(chip);
    });

    resultPanel.classList.remove('hidden');
  } catch (err) {
    statusMsg.textContent = 'Kuch ghalat ho gaya, dobara try karein.';
  } finally {
    analyzeBtn.disabled = false;
  }
});

### `app.py` — Flask backend (prediction + influential keywords)

In [ ]:
%%writefile /content/app.py
from flask import Flask, jsonify, request, render_template
import pickle
import re
import string
import numpy as np
from nltk.corpus import stopwords

app = Flask(__name__, template_folder="templates", static_folder="static")

with open("model.pkl", "rb") as f:
    model = pickle.load(f)
with open("vectorizer.pkl", "rb") as f:
    tfidf = pickle.load(f)

STOPWORDS = set(stopwords.words("english"))
FEATURE_NAMES = np.array(tfidf.get_feature_names_out())

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(words)

def get_top_keywords(vec, pred_label, top_n=8):
    if not hasattr(model, "coef_"):
        return []
    coefs = model.coef_[0]
    nonzero_idx = vec.nonzero()[1]
    if len(nonzero_idx) == 0:
        return []
    scores = coefs[nonzero_idx] * vec[0, nonzero_idx].toarray()[0]
    order = np.argsort(scores)
    if pred_label == 1:   # FAKE -> largest positive contributions
        top_idx = nonzero_idx[order[::-1][:top_n]]
    else:                 # REAL -> largest negative contributions
        top_idx = nonzero_idx[order[:top_n]]
    return [FEATURE_NAMES[i] for i in top_idx]

@app.route("/")
def home():
    return render_template("index.html")

@app.route("/api/predict", methods=["POST"])
def api_predict():
    data = request.get_json(force=True)
    text = (data or {}).get("text", "").strip()

    if not text:
        return jsonify({"error": "Koi text nahi mila."}), 400

    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned])
    pred = int(model.predict(vec)[0])

    if hasattr(model, "predict_proba"):
        confidence = float(model.predict_proba(vec)[0][pred])
    elif hasattr(model, "decision_function"):
        score = float(model.decision_function(vec)[0])
        confidence = 1 / (1 + np.exp(-abs(score)))
    else:
        confidence = 1.0

    label = "FAKE" if pred == 1 else "REAL"
    keywords = get_top_keywords(vec, pred)

    return jsonify({
        "label": label,
        "confidence": round(confidence * 100, 2),
        "keywords": keywords
    })

if __name__ == "__main__":
    app.run(port=5000)

## Step 11: Use ngrok API here for showing interface on web

In [ ]:
!pip install flask pyngrok -q

from getpass import getpass
from pyngrok import ngrok

ngrok_token = getpass("3GJftbpa7YGYVMWlst2xFkoyCzL_7NEHhdeRB6hbMjfpZzhgW")
ngrok.set_auth_token(ngrok_token.strip())
print("ngrok authtoken set ho gaya ✅")

In [ ]:
import subprocess, time

ngrok.kill()

flask_process = subprocess.Popen(["python", "/content/app.py"], cwd="/content")
time.sleep(4)

public_url = ngrok.connect(5000)
print("🎉 Aapka interface yahan open hai:")
print(public_url)

In [ ]:
ngrok.kill()
flask_process.terminate()
print("App Closed")